# 03 — Structural curation audit

**Workflow version:** 0.5.1

Audit the released original and curated SBML models directly. This notebook is independent of phenotype accuracy and asks whether reaction bounds, GPRs, added reactions, and selected stoichiometric edits changed as expected. It also targets the three phenotype regressions observed in notebook 02.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from cobra.io import read_sbml_model

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = ROOT / "data" / "raw"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

original = read_sbml_model(str(DATA_DIR / "yeast9.0.xml"))
curated = read_sbml_model(str(DATA_DIR / "Yeast9_curated.xml"))

PAIRWISE_PATH = RESULTS_DIR / "02_pairwise_comparison.csv"
if not PAIRWISE_PATH.exists():
    raise FileNotFoundError("Run notebook 02 first so 02_pairwise_comparison.csv is available.")

pairwise = pd.read_csv(PAIRWISE_PATH)

In [ ]:
CURATION_REACTIONS = [
    "r_0217", "r_0172",
    "r_2488", "r_2489", "r_2490", "r_2491",
    "r_2492", "r_2493", "r_2494", "r_2495",
    "r_0312", "r_4702", "r_4703", "r_0559",
    "r_4048", "r_4598", "r_1026", "r_0080", "r_0815",
    "r_2067", "r_2070", "r_2071", "r_0250",
    "r_temp1", "r_temp2", "r_temp3",
]

def snapshot(model, reaction_id):
    if reaction_id not in model.reactions:
        return {
            "reaction": reaction_id, "present": False,
            "lower_bound": np.nan, "upper_bound": np.nan,
            "gpr": "", "reaction_name": "", "equation": "",
        }
    reaction = model.reactions.get_by_id(reaction_id)
    return {
        "reaction": reaction_id, "present": True,
        "lower_bound": reaction.lower_bound,
        "upper_bound": reaction.upper_bound,
        "gpr": reaction.gene_reaction_rule,
        "reaction_name": reaction.name,
        "equation": reaction.reaction,
    }

rows = []
for reaction_id in CURATION_REACTIONS:
    rows.append({"model": "Yeast9", **snapshot(original, reaction_id)})
    rows.append({"model": "Yeast9_curated", **snapshot(curated, reaction_id)})

structural_audit = pd.DataFrame(rows)
display(structural_audit)
structural_audit.to_csv(RESULTS_DIR / "03_structural_curation_audit.csv", index=False)

## Regression-focused structural audit

Notebook 02 produced three curated-model regressions. This section records their phenotype values and inspects the gene-reaction structure around the affected arginine/uracil and thiamine cases. The tables are evidence for diagnosis; they do not by themselves establish causality.


In [ ]:
regression_columns = [
    "pair_id",
    "excel_row_original",
    "gene_field_original",
    "chemical_original",
    "classification_original",
    "classification_curated",
    "ko_growth_original",
    "ko_growth_curated",
    "rescue_growth_original",
    "rescue_growth_curated",
    "mapped_background_original",
    "mapped_background_curated",
    "mapped_rescue_original",
    "mapped_rescue_curated",
]

regressions = pairwise.loc[pairwise["change"] == "regression", regression_columns].copy()

if len(regressions) != 3:
    raise AssertionError(f"Expected 3 regressions from notebook 02, found {len(regressions)}.")

display(regressions)
regressions.to_csv(RESULTS_DIR / "03_regression_pairs.csv", index=False)


In [ ]:
FOCUS_GENES = ["YOR303W", "YJR109C", "YJL130C", "YPL214C", "YGR144W"]

def gene_reaction_snapshot(model, model_label, gene_id):
    if gene_id not in model.genes:
        return [{
            "model": model_label,
            "gene": gene_id,
            "reaction": "",
            "present": False,
            "lower_bound": np.nan,
            "upper_bound": np.nan,
            "gpr": "",
            "equation": "",
        }]

    gene = model.genes.get_by_id(gene_id)
    reactions = sorted(gene.reactions, key=lambda reaction: reaction.id)

    if not reactions:
        return [{
            "model": model_label,
            "gene": gene_id,
            "reaction": "",
            "present": True,
            "lower_bound": np.nan,
            "upper_bound": np.nan,
            "gpr": "",
            "equation": "",
        }]

    return [
        {
            "model": model_label,
            "gene": gene_id,
            "reaction": reaction.id,
            "present": True,
            "lower_bound": reaction.lower_bound,
            "upper_bound": reaction.upper_bound,
            "gpr": reaction.gene_reaction_rule,
            "equation": reaction.reaction,
        }
        for reaction in reactions
    ]

gene_rows = []
for gene_id in FOCUS_GENES:
    gene_rows.extend(gene_reaction_snapshot(original, "Yeast9", gene_id))
    gene_rows.extend(gene_reaction_snapshot(curated, "Yeast9_curated", gene_id))

gene_reaction_audit = pd.DataFrame(gene_rows)
display(gene_reaction_audit)
gene_reaction_audit.to_csv(RESULTS_DIR / "03_regression_gene_reaction_audit.csv", index=False)


In [ ]:
REGRESSION_FOCUS_REACTIONS = [
    "r_0250",
    "r_2067",
    "r_2070",
    "r_2071",
    "r_temp2",
    "r_temp3",
]

regression_reaction_audit = structural_audit.loc[
    structural_audit["reaction"].isin(REGRESSION_FOCUS_REACTIONS),
    [
        "model",
        "reaction",
        "present",
        "lower_bound",
        "upper_bound",
        "gpr",
        "reaction_name",
        "equation",
    ],
].copy()

display(regression_reaction_audit)
regression_reaction_audit.to_csv(
    RESULTS_DIR / "03_regression_reaction_audit.csv",
    index=False,
)


In [ ]:
def stoichiometry_by_id(reaction):
    return {metabolite.id: float(coefficient) for metabolite, coefficient in reaction.metabolites.items()}

def stoichiometric_diff(original_model, curated_model, reaction_id):
    if reaction_id not in original_model.reactions or reaction_id not in curated_model.reactions:
        return pd.DataFrame()
    a = stoichiometry_by_id(original_model.reactions.get_by_id(reaction_id))
    b = stoichiometry_by_id(curated_model.reactions.get_by_id(reaction_id))
    metabolite_ids = sorted(set(a) | set(b))
    rows = []
    for metabolite_id in metabolite_ids:
        old = a.get(metabolite_id, 0.0)
        new = b.get(metabolite_id, 0.0)
        if not np.isclose(old, new, atol=1e-12):
            rows.append({
                "reaction": reaction_id,
                "metabolite": metabolite_id,
                "original_coefficient": old,
                "curated_coefficient": new,
                "delta": new - old,
            })
    return pd.DataFrame(rows)

stoich_changes = pd.concat(
    [stoichiometric_diff(original, curated, rid) for rid in ["r_4048", "r_4598"]],
    ignore_index=True,
)

display(stoich_changes)
stoich_changes.to_csv(RESULTS_DIR / "03_stoichiometric_changes.csv", index=False)

In [ ]:
focus = structural_audit[structural_audit["reaction"].isin(["r_0217", "r_0172", "r_4702", "r_4703", "r_0080", "r_0250", "r_temp1", "r_temp2", "r_temp3"])]
display(
    focus.pivot(
        index="reaction",
        columns="model",
        values=["present", "lower_bound", "upper_bound", "gpr"],
    )
)

## Interpretation checkpoint

This notebook reports **released-artifact structure**. A mismatch between the paper description and the released curated SBML is an artifact-level reproducibility observation; it is not by itself evidence about author intent.